# ETL Pipeline Development with Pandas and scikit-learn

This notebook demonstrates a complete ETL pipeline that loads raw data, preprocesses it, transforms it with scikit-learn, and saves the results to destination files.

## 1. Import Required Libraries

Import pandas, numpy, and scikit-learn utilities used for preprocessing and transformation.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## 2. Load Raw Data

Create or read a raw CSV dataset using pandas. This section simulates raw input data and then loads it from disk.

In [ ]:
raw_data_path = Path('raw_data.csv')

raw_df = pd.DataFrame({
    'customer_id': [101, 102, 103, 104, 105, 106],
    'age': [34, np.nan, 45, 23, 35, 40],
    'income': [55000, 64000, 72000, np.nan, 58000, 60000],
    'gender': ['Male', 'Female', 'Female', 'Male', None, 'Female'],
    'region': ['North', 'South', 'East', 'West', 'North', 'South'],
    'purchased': [1, 0, 1, 0, 1, 0]
})

raw_df.to_csv(raw_data_path, index=False)
loaded_df = pd.read_csv(raw_data_path)
loaded_df.head()

## 3. Preprocess Data

Handle missing values, drop duplicates, and clean data before transformation.

In [ ]:
def preprocess_data(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df = df.drop_duplicates()
    df['gender'] = df['gender'].fillna('Unknown')
    df['age'] = df['age'].fillna(df['age'].median())
    df['income'] = df['income'].fillna(df['income'].median())
    df['region'] = df['region'].astype('category')
    return df

preprocessed_df = preprocess_data(loaded_df)
preprocessed_df

## 4. Transform Data

Apply feature scaling and encoding using scikit-learn preprocessing tools. This step creates a reusable transformation pipeline.

In [ ]:
numeric_features = ['age', 'income']
categorical_features = ['gender', 'region']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features),
    ],
    remainder='drop'
)

def transform_data(df: pd.DataFrame) -> pd.DataFrame:
    transformed_array = preprocessor.fit_transform(df)
    numeric_cols = numeric_features
    categorical_cols = preprocessor.named_transformers_['cat']['onehot'].get_feature_names_out(categorical_features)
    feature_names = list(numeric_cols) + list(categorical_cols)
    transformed_df = pd.DataFrame(transformed_array, columns=feature_names)
    transformed_df['customer_id'] = df['customer_id'].values
    transformed_df['purchased'] = df['purchased'].values
    return transformed_df

transformed_df = transform_data(preprocessed_df)
transformed_df.head()

## 5. Load Data to Destination

Save the processed output to destination files using pandas.

In [ ]:
output_csv_path = Path('processed_data.csv')
output_parquet_path = Path('processed_data.parquet')

def load_data(df: pd.DataFrame, csv_path: Path, parquet_path: Path) -> None:
    df.to_csv(csv_path, index=False)
    df.to_parquet(parquet_path, index=False)

load_data(transformed_df, output_csv_path, output_parquet_path)
print(f'Processed CSV saved to: {output_csv_path.resolve()}')
print(f'Processed Parquet saved to: {output_parquet_path.resolve()}')

## 6. Automate the ETL Pipeline

Wrap the ETL steps into reusable functions and execute them in sequence.

In [ ]:
def run_etl(raw_path: Path, csv_out: Path, parquet_out: Path) -> pd.DataFrame:
    df = pd.read_csv(raw_path)
    df = preprocess_data(df)
    transformed = transform_data(df)
    load_data(transformed, csv_out, parquet_out)
    return transformed

final_df = run_etl(raw_data_path, output_csv_path, output_parquet_path)
final_df.head()